In [42]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os

In [43]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/macbook/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [44]:
apps_df = pd.read_csv('googleplaystore.csv')
reviews_df = pd.read_csv('User Reviews.csv')  # keep this if you also have the reviews file

apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [45]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [46]:
#Step 2 : Data Cleaning
apps_df = apps_df.dropna(subset=['Rating'])
for column in apps_df.columns :
    apps_df[column].fillna(apps_df[column].mode()[0], inplace=True)

apps_df.drop_duplicates(inplace=True)

apps_df = apps_df = apps_df[apps_df['Rating'] <= 5]

reviews_df.dropna(subset=['Translated_Review'], inplace=True)

/var/folders/w7/7tsg0lcs29sb7qy3zbqr507m0000gn/T/ipykernel_3191/1166397953.py:4: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





In [47]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [48]:
#Convert the Installs columns to numeric by removing commas and +
apps_df['Installs'] = apps_df['Installs'].str.replace(',', '').str.replace('+', '').astype(int)

#Convert Price column to numeric after removing $
apps_df['Price'] = apps_df['Price'].str.replace('$', '').astype(float)

In [49]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs            int64
Type               object
Price             float64
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [50]:
merged_df = pd.merge(apps_df, reviews_df, on='App', how='inner')

merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I hate,Negative,-0.800,0.900000


In [51]:
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M', ''))
    elif 'k' in size:
        return float(size.replace('k', '')) / 1024
    else:
        return np.nan

apps_df['Size'] = apps_df['Size'].apply(convert_size)

apps_df

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10834,FR Calculator,FAMILY,4.0,7,2.6,500,Free,0.0,Everyone,Education,"June 18, 2017",1.0.0,4.1 and up
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53.0,5000,Free,0.0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6,100,Free,0.0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,NaN,1000,Free,0.0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


In [52]:
#Lograrithmic
apps_df['Log_Installs'] = np.log(apps_df['Installs'])

apps_df['Reviews'] = apps_df['Reviews'].astype(int)

apps_df['Log_Reviews'] = np.log(apps_df['Reviews'])

apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews             int64
Size              float64
Installs            int64
Type               object
Price             float64
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
Log_Installs      float64
Log_Reviews       float64
dtype: object

In [53]:
def rating_group(rating):
    if rating >= 4:
        return 'Top rated app'
    elif rating >= 3:
        return 'Above average'
    elif rating >= 2:
        return 'Average'
    else:
        return 'Below Average'

apps_df['Rating_Group'] = apps_df['Rating'].apply(rating_group)

apps_df['Revenue'] = apps_df['Price'] * apps_df['Installs']

In [54]:
#Revenue column
apps_df['Revenue'] = apps_df['Price'] * apps_df['Installs']

sia = SentimentIntensityAnalyzer()

#Polarity Scores in SIA
#Positive, Negative, Neutral and Compound: -1 = Very negative ; +1 = Very positive

review = "This app is amazing! I love the new features."
sentiment_score = sia.polarity_scores(review)
print(sentiment_score)

review = "This app is very bad! I hate the new features."
sentiment_score = sia.polarity_scores(review)
print(sentiment_score)

review = "This app is okay."
sentiment_score = sia.polarity_scores(review)
print(sentiment_score)


reviews_df['Sentiment_Score'] = reviews_df['Translated_Review'].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

reviews_df.head()

{'neg': 0.0, 'neu': 0.42, 'pos': 0.58, 'compound': 0.8516}
{'neg': 0.535, 'neu': 0.465, 'pos': 0.0, 'compound': -0.8427}
{'neg': 0.0, 'neu': 0.612, 'pos': 0.388, 'compound': 0.2263}


,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Sentiment_Score
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333,0.9531
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462,0.6597
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000,0.6249
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000,0.6369
5,10 Best Foods for You,Best way,Positive,1.00,0.300000,0.6369


In [55]:
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')

apps_df['Year'] = apps_df['Last Updated'].dt.year

apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Year
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,9.210340,5.068904,Top rated app,0.0,2018
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,13.122363,6.874198,Above average,0.0,2018
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,15.424948,11.379508,Top rated app,0.0,2018
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,17.727534,12.281384,Top rated app,0.0,2018
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,11.512925,6.874198,Top rated app,0.0,2018


In [56]:
import nltk

In [57]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/macbook/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [58]:
html_files_path = "./"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

In [59]:
plot_containers = ""

def save_plot_as_html(fig, filename, insight):
    global plot_containers
    filepath = os.path.join(html_files_path, filename)
    html_content = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    
    plot_containers += f"""
    <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
    
    fig.write_html(filepath, full_html=False, include_plotlyjs='inline')

plot_width = 400
plot_height = 300
plot_bg_color = "black"
text_color = "white"
title_font = {'size': 16}
axis_font = {'size': 12}


# Figure 1
category_counts = apps_df['Category'].value_counts().nlargest(10)

fig1 = px.bar(
    x=category_counts.index,
    y=category_counts.values,
    labels={'x': 'Category', 'y': 'Count'},
    title='Top Categories on Play Store',
    color=category_counts.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=400,
    height=300
)

fig1.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size': 16},
    xaxis=dict(title_font={'size': 12}),
    yaxis=dict(title_font={'size': 12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

#fig1.update_traces(marker=dict(pattern=dict(line=dict(color='white', width=1))))

save_plot_as_html(
    fig1,
    "Category Graph 1.html",
    "The top categories on the Play Store are dominated by tools, entertainment, and productivity apps"
)

In [60]:
#Figure 2
type_counts = apps_df['Type'].value_counts()

fig2 = px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title='App Type Distribution',
    color_discrete_sequence=px.colors.sequential.RdBu,
    width=400,
    height=300
)

fig2.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size': 16},
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(
    fig2,
    "Type Graph 2.html",
    "Most apps on the Playstore are free, indicating a strategy to attract users first and monetize through ads or in-app purchases."
)

In [61]:
#Figure 3
fig3 = px.histogram(
    apps_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=['#636EFA'],
    width=400,
    height=300
)

fig3.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size': 16},
    xaxis=dict(title_font={'size': 12}),
    yaxis=dict(title_font={'size': 12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(
    fig3,
    "Rating Graph 3.html",
    "Ratings are skewed towards higher values, suggesting that most apps are rated favorably by users"
)

In [62]:
#Figure 4
sentiment_counts = reviews_df['Sentiment_Score'].value_counts()

fig4 = px.bar(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    labels={'x': 'Sentiment Score', 'y': 'Count'},
    title='Sentiment Distribution',
    color=sentiment_counts.index,
    color_discrete_sequence=px.colors.sequential.RdPu,
    width=400,
    height=300
)

fig4.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size': 16},
    xaxis=dict(title_font={'size': 12}),
    yaxis=dict(title_font={'size': 12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(
    fig4,
    "Sentiment Graph 4.html",
    "Sentiments in reviews show a mix of positive and negative feedback, with a slight lean towards positive sentiment."
)

In [63]:
#Figure 5
installs_by_category = apps_df.groupby('Category')['Installs'].sum().nlargest(10)

fig5 = px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    orientation='h',
    labels={'x': 'Installs', 'y': 'Category'},
    title='Installs by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Blues,
    width=400,
    height=300
)

fig5.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size': 16},
    xaxis=dict(title_font={'size': 12}),
    yaxis=dict(title_font={'size': 12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(
    fig5,
    "Installs Graph 5.html",
    "The categories with the most installs are social and communication apps, reflecting their broad appeal and high user demand."
)

In [64]:
# Updates Per Year Plot
updates_per_year = apps_df['Last Updated'].dt.year.value_counts().sort_index()

fig6 = px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    labels={'x': 'Year', 'y': 'Number of Updates'},
    title='Number of Updates Over the Years',
    color_discrete_sequence=['#AB63FA'],
    width=plot_width,
    height=plot_height
)

fig6.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(
    fig6,
    "updates_per_year.html",
    "Updates have been increasing over the years, showing that developers are actively maintaining and improving their applications."
)

In [65]:
#Figure 7
revenue_by_category = apps_df.groupby('Category')['Revenue'].sum().nlargest(10)

fig7 = px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    labels={'x': 'Category', 'y': 'Revenue'},
    title='Revenue by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=400,
    height=300
)

fig7.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size': 16},
    xaxis=dict(title_font={'size': 12}),
    yaxis=dict(title_font={'size': 12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(
    fig7,
    "Revenue Graph 7.html",
    "Categories such as Business and Productivity lead in revenue generation, indicating their monetization potential."
)

In [66]:
#Figure 8
genre_counts = apps_df['Genres'].str.split(';', expand=True).stack().value_counts().nlargest(10)

fig8 = px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x': 'Genre', 'y': 'Count'},
    title='Top Genres',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.OrRd,
    width=400,
    height=300
)

fig8.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size': 16},
    xaxis=dict(title_font={'size': 12}),
    yaxis=dict(title_font={'size': 12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(
    fig8,
    "Genre Graph 8.html",
    "Action and Casual genres are the most common, reflecting user preferences and market trends."
)

In [67]:
#Figure 9
fig9 = px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=400,
    height=300
)

fig9.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size': 16},
    xaxis=dict(title_font={'size': 12}),
    yaxis=dict(title_font={'size': 12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(
    fig9,
    "Update Graph 8.html",
    "The Scatter Plot shows a weak correlation between the last update and ratings."
)

In [68]:
#Figure 10
fig10 = px.box(
    apps_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Rating for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=400,
    height=300
)

fig10.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size': 16},
    xaxis=dict(title_font={'size': 12}),
    yaxis=dict(title_font={'size': 12}),
    margin=dict(l=10, r=10, t=30, b=10)
)

save_plot_as_html(
    fig10,
    "Paid Free Graph 10.html",
    "Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from paid applications."
)

In [69]:
plot_containers_split=plot_containers.split('</div>')

In [70]:
if len(plot_containers_split) > 1:
    final_plot = plot_containers_split[-2] + '</div>'
else:
    final_plot = plot_containers

In [71]:
dashboard_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width,initial-scale=1.0">
    <title> Google Play Store Review Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555;
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0,0,0,0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container:hover .insights {{
            display: block;
        }}
    </style>
    <script>
        function openPlot(filename) {{
            window.open(filename, '_blank');
        }}
    </script>
</head>
<body>
    <div class="header">
        <img src="https://www.google.com/images/branding/googlelogo/2x/googlelogo_light_color_272x92dp.png" alt="Google Logo">
        <h1>Google Play Store Reviews Analytics</h1>
        <img src="https://upload.wikimedia.org/wikipedia/commons/7/78/Google_Play_Store_badge_EN.svg" alt="Play Store Logo">
    </div>
    <div class="container">
        {plots}
    </div>
</body>
</html>
"""

In [72]:
final_html = dashboard_html.format(
    plots=plot_containers,
    plot_width=plot_width,
    plot_height=plot_height
)

dashboard_path = os.path.join(html_files_path, "web page.html")

with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)

webbrowser.open('file:///' + os.path.realpath(dashboard_path))

True

In [73]:
# =============================
# Internship Tasks
# =============================

In [74]:
# ==============================
# TASK 1
# ==============================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime
import pytz
import os
import webbrowser

# Initialize container ONLY ONCE
plot_containers = ""

# ==============================
# DATA PREPARATION
# ==============================

# Convert date column
apps_df['Last Updated'] = pd.to_datetime(
    apps_df['Last Updated'],
    errors='coerce'
)

# Apply filters
filtered_df = apps_df[
    (apps_df['Rating'] >= 4.0) &
    (apps_df['Size'] >= 10) &
    (apps_df['Last Updated'].dt.month == 1)
]

# Top 10 categories by installs
top_categories = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

# Filter top categories
top_df = filtered_df[
    filtered_df['Category'].isin(top_categories)
]

# Aggregation
summary = top_df.groupby('Category').agg({
    'Rating': 'mean',
    'Reviews': 'sum'
}).reset_index()

# ==============================
# TIME CONDITION
# ==============================

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)

if 15 <= current_time.hour < 17:

    # ==============================
    # CREATE FIGURE
    # ==============================

    fig1 = go.Figure()

    # Average Rating Bar
    fig1.add_bar(
        x=summary['Category'],
        y=summary['Rating'],
        name='Average Rating'
    )

    # Total Reviews Bar
    fig1.add_bar(
        x=summary['Category'],
        y=summary['Reviews'],
        name='Total Reviews',
        yaxis='y2'
    )

    # ==============================
    # LAYOUT
    # ==============================

    fig1.update_layout(
        title="Top 10 Categories by Installs",

        xaxis=dict(
            title="Category"
        ),

        yaxis=dict(
            title="Average Rating"
        ),

        yaxis2=dict(
            title="Total Reviews",
            overlaying='y',
            side='right',
            showticklabels=False
        ),

        barmode='group',
        template='plotly_dark',
        width=1200,
        height=600
    )

    # Show chart
    fig1.show()

    # ==============================
    # SAVE CHART AS HTML
    # ==============================

    task1_file = "Task1_Top_Categories.html"

    fig1.write_html(
        os.path.join(html_files_path, task1_file),
        full_html=True
    )

    # ==============================
    # ADD CHART TO DASHBOARD
    # ==============================

    plot_containers += f"""
    <div class="plot-container"
         onclick="openPlot('{task1_file}')">

        <iframe
            src="{task1_file}"
            width="100%"
            height="100%"
            frameborder="0">
        </iframe>

        <div class="insights">
            <strong>Task 1 Insights:</strong><br>
            Top app categories show strong installs and reviews.<br>
            Highly rated categories attract more engagement.
        </div>

    </div>
    """

else:
    print("Chart available only between 3 PM and 5 AM IST")

Chart available only between 3 PM and 5 AM IST


In [75]:
# ==============================
# Task 2 WITH HTML DASHBOARD
# ==============================

import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime
import pytz
import os

df = pd.read_csv('googleplaystore.csv')

# ==============================
# CLEAN INSTALLS
# ==============================

df['Installs'] = df['Installs'].str.replace('[+,]', '', regex=True)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

# ==============================
# REMOVE UNWANTED CATEGORIES
# ==============================

df = df[
    ~df['Category'].str.startswith(
        ('A', 'C', 'G', 'S'),
        na=False
    )
]

# ==============================
# TOP 5 CATEGORIES
# ==============================

top5 = (
    df.groupby('Category')['Installs']
      .sum()
      .nlargest(5)
      .index
)

df_top = df[df['Category'].isin(top5)]

# ==============================
# GROUP DATA
# ==============================

category_data = (
    df_top.groupby('Category')['Installs']
          .sum()
          .reset_index()
)

# ==============================
# HIGHLIGHT
# ==============================

category_data['Highlight'] = np.where(
    category_data['Installs'] > 1_000_000,
    'Above 1M',
    'Below 1M'
)

# ==============================
# ISO COUNTRY CODES
# ==============================

iso_codes = ['USA', 'IND', 'BRA', 'CAN', 'AUS']
category_data['ISO'] = iso_codes[:len(category_data)]

# ==============================
# TIME CHECK
# ==============================

ist = pytz.timezone('Asia/Kolkata')
current_hour = datetime.now(ist).hour

# ==============================
# CREATE MAP
# ==============================

if 18 <= current_hour < 20:

    fig2 = px.choropleth(
        category_data,
        locations='ISO',
        color='Installs',
        hover_name='Category',
        color_continuous_scale='Blues',
        title='Global Installs by Top 5 App Categories',
    )

    # ==============================
    # FIX COLOR SCALE
    # ==============================

    fig2.update_layout(
        coloraxis_colorbar=dict(
            title='Installs',
            tickformat='.2s'
        ),
        geo=dict(
            bgcolor='white'
        ),
        paper_bgcolor='white',
        plot_bgcolor='white'
    )

    fig2.update_traces(marker_line_width=2)

    # ==============================
    # SHOW MAP
    # ==============================

    fig2.show()

    # ==============================
    # SAVE HTML
    # ==============================

    task2_file = "Task2_Global_Installs.html"

    fig2.write_html(
        os.path.join(html_files_path, task2_file),
        full_html=False,
        include_plotlyjs='cdn'
    )

    # ==============================
    # ADD TO DASHBOARD
    # ==============================

    if task2_file not in plot_containers:

        plot_containers += f"""
        <div class="plot-container" onclick="openPlot('{task2_file}')">

            <iframe
                src="{task2_file}"
                width="100%"
                height="100%"
                frameborder="0">
            </iframe>

            <div class="insights">
                <strong>Task 2 Insights:</strong><br>
                Global map shows installs across top app categories.<br>
                Color intensity represents higher install counts.
            </div>

        </div>
        """

else:
    print("Map visible only between 6 PM and 8 AM IST")

# ==============================
# OPTIONAL COLOR COLUMN
# ==============================

category_data['Color'] = category_data['Highlight'].map({
    'Above 1M': 'red',
    'Below 1M': 'blue'
})

Map visible only between 6 PM and 8 AM IST


In [76]:
# INSTALL mpld3 FIRST
# Run this line once in terminal or notebook

!pip install mpld3

In [77]:
# =========================================================
# TASK 3
# Free vs Paid Apps – Avg Installs & Revenue
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz
import mpld3
import os

# =========================================================
# TIME CONDITION
# =========================================================

ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)

if 13 <= current_time.hour < 14:

    df = apps_df.copy()

    # =====================================================
    # DATA CLEANING
    # =====================================================

    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
    df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

    df['Android Ver'] = (
        df['Android Ver']
        .astype(str)
        .str.extract(r'(\d+\.\d+)')
    )

    df['Android Ver'] = pd.to_numeric(
        df['Android Ver'],
        errors='coerce'
    )

    # =====================================================
    # REVENUE
    # =====================================================

    df['Revenue'] = df['Installs'] * df['Price']

    # =====================================================
    # FILTERS
    # =====================================================

    df = df[
        (df['Installs'] >= 10000) &
        (df['Android Ver'] > 4.0) &
        (df['Size'] > 15) &
        (df['Content Rating'] == 'Everyone') &
        (df['App'].str.len() <= 30) &
        (
            ((df['Type'] == 'Paid') & (df['Revenue'] >= 10000)) |
            (df['Type'] == 'Free')
        )
    ]

    # =====================================================
    # TOP 3 CATEGORIES
    # =====================================================

    top_categories = (
        df.groupby('Category')['Installs']
        .sum()
        .sort_values(ascending=False)
        .head(3)
        .index
    )

    df = df[df['Category'].isin(top_categories)]

    # =====================================================
    # AGGREGATION
    # =====================================================

    result = (
        df.groupby(['Category', 'Type'])[['Installs', 'Revenue']]
        .mean()
        .reset_index()
    )

    # =====================================================
    # PLOT
    # =====================================================

    fig, ax1 = plt.subplots(figsize=(12, 7))
    ax2 = ax1.twinx()

    categories = result['Category'].unique()
    x = np.arange(len(categories))
    bar_width = 0.35

    for i, t in enumerate(['Free', 'Paid']):

        temp = result[result['Type'] == t]

        ax1.bar(
            x + (i - 0.5) * bar_width,
            temp['Installs'],
            width=bar_width,
            label=f'{t} Installs'
        )

        ax2.bar(
            x + (i - 0.5) * bar_width,
            temp['Revenue'],
            width=bar_width,
            alpha=0.5,
            label=f'{t} Revenue'
        )

    # =====================================================
    # LABELS
    # =====================================================

    ax1.set_xticks(x)
    ax1.set_xticklabels(categories)

    ax1.set_ylabel("Average Installs")
    ax2.set_ylabel("Average Revenue ($)")

    ax1.set_title(
        "Free vs Paid Apps – Avg Installs & Revenue (Top 3 Categories)"
    )

    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')

    plt.tight_layout()

    # =====================================================
    # SHOW GRAPH
    # =====================================================

    plt.show()

    # =====================================================
    # SAVE HTML
    # =====================================================

    task3_file = "task3_dashboard.html"

    html_str = mpld3.fig_to_html(fig)

    html_str = f"""
    <html>
    <head>
        <style>

            body {{
                margin: 0;
                padding: 20px;
                background: white;
            }}

            svg {{
                overflow: visible !important;
            }}

            .mpld3-figure {{
                margin: auto;
            }}

        </style>
    </head>

    <body>
        {html_str}
    </body>
    </html>
    """

    with open(
        os.path.join(html_files_path, task3_file),
        "w",
        encoding="utf-8"
    ) as f:

        f.write(html_str)

    print("HTML file saved successfully!")

    # =====================================================
    # ADD TO DASHBOARD
    # =====================================================

    plot_containers += f"""

    <div class="plot-container">

        <iframe
            src="{task3_file}"
            width="100%"
            height="500"
            frameborder="0">
        </iframe>

        <div class="insights">
            <strong>Task 3 Insights:</strong><br>
            Comparison between Free and Paid apps based on
            installs and revenue across top categories.
        </div>

    </div>

    """

else:
    print("Graph available only between 1 PM and 2 AM IST")

Graph available only between 1 PM and 2 AM IST


In [78]:
# ==============================
# Task 4 – Time Series with Growth Highlight
# WITH HTML DASHBOARD PART
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz
import mpld3
import os

# Time restriction (6 PM – 9 PM IST)
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)

if 18 <= current_time.hour < 21:

    df = apps_df.copy()

    # --- Basic cleaning ---
    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
    df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

    # Convert date column
    df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')
    df['Month'] = df['Last Updated'].dt.to_period('M')

    # --- Apply Filters ---
    df = df[
        (df['Reviews'] > 500) &
        (~df['App'].str.startswith(('X','Y','Z'))) &
        (~df['App'].str.contains('S', case=False, na=False)) &
        (df['Category'].str.startswith(('E','C','B')))
    ]

    # --- Translate Categories ---
    category_translation = {
        'BEAUTY': 'सौंदर्य',
        'BUSINESS': 'வணிகம்',
        'DATING': 'Dating (Deutsch)'
    }

    df['Category'] = df['Category'].replace(category_translation)

    # --- Aggregate installs monthly by category ---
    monthly_data = (
        df.groupby(['Month', 'Category'])['Installs']
          .sum()
          .reset_index()
    )

    monthly_data['Month'] = monthly_data['Month'].dt.to_timestamp()

    plt.rcParams['font.family'] = 'Arial Unicode MS'

    # --- Plot ---
    plt.figure(figsize=(10,6))

    for category in monthly_data['Category'].unique():

        cat_data = monthly_data[monthly_data['Category'] == category]
        cat_data = cat_data.sort_values('Month')

        plt.plot(
            cat_data['Month'],
            cat_data['Installs'],
            label=category,
            linewidth=2
        )

        # Calculate MoM growth
        cat_data['MoM_Growth'] = cat_data['Installs'].pct_change()

        # Highlight >20% growth
        high_growth = cat_data['MoM_Growth'] > 0.20

        plt.fill_between(
            cat_data['Month'],
            cat_data['Installs'],
            where=high_growth,
            alpha=0.3,
            color='orange',
            interpolate=True
        )

    plt.xlabel("Time (Month)")
    plt.ylabel("Total Installs")
    plt.title("Monthly Install Trend by Category (Growth >20% Highlighted)")
    plt.legend(fontsize=9, loc='upper left')
    plt.tight_layout()
    plt.xticks(rotation=45)

    # ==============================
    # SAVE HTML
    # ==============================

    task4_file = "Task4_TimeSeries_Growth.html"

    html_str = mpld3.fig_to_html(plt.gcf())

    with open(os.path.join(html_files_path, task4_file), "w", encoding="utf-8") as f:
        f.write(html_str)

    # ==============================
    # ADD TO DASHBOARD
    # ==============================

    plot_containers += f"""
    <div class="plot-container" onclick="openPlot('{task4_file}')">

        <iframe
            src="{task4_file}"
            width="100%"
            height="100%"
            frameborder="0">
        </iframe>

        <div class="insights">
            <strong>Task 4 Insights:</strong><br>
            Monthly install trends reveal category growth patterns.<br>
            Orange highlighted regions indicate more than 20% MoM growth.
        </div>

    </div>
    """

    # Show graph
    plt.show()

else:
    print("Graph available only between 6 PM and 9 PM IST")

Graph available only between 6 PM and 9 PM IST


In [79]:
# ==============================
# TASK 5 WITH HTML DASHBOARD
# ==============================

import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime
import pytz
import os

df = pd.read_csv('googleplaystore.csv')

# ==============================
# CLEAN SIZE
# ==============================

df['Size'] = df['Size'].str.replace('M', '', regex=False)
df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

# ==============================
# CLEAN INSTALLS
# ==============================

df['Installs'] = df['Installs'].str.replace('[+,]', '', regex=True)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

# ==============================
# CLEAN REVIEWS
# ==============================

df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

# ==============================
# SIMULATE SENTIMENT SUBJECTIVITY
# ==============================

np.random.seed(0)
df['Sentiment_Subjectivity'] = np.random.uniform(0, 1, len(df))

# ==============================
# APPLY FILTERS
# ==============================

valid_categories = [
    'GAME', 'BEAUTY', 'BUSINESS', 'COMICS',
    'COMMUNICATION', 'DATING', 'ENTERTAINMENT',
    'SOCIAL', 'EVENTS'
]

df_filtered = df[
    (df['Rating'] > 3.5) &
    (df['Reviews'] > 500) &
    (df['Installs'] > 50000) &
    (df['Sentiment_Subjectivity'] > 0.5) &
    (df['Category'].isin(valid_categories)) &
    (~df['App'].str.contains('S', case=False, na=False))
].copy()

# ==============================
# CATEGORY TRANSLATION
# ==============================

df_filtered['Category_Translated'] = df_filtered['Category']

df_filtered.loc[
    df_filtered['Category'] == 'BEAUTY',
    'Category_Translated'
] = 'सौंदर्य'

df_filtered.loc[
    df_filtered['Category'] == 'BUSINESS',
    'Category_Translated'
] = 'வணிகம்'

df_filtered.loc[
    df_filtered['Category'] == 'DATING',
    'Category_Translated'
] = 'Dating (DE)'

# ==============================
# TIME CONDITION
# ==============================

ist = pytz.timezone('Asia/Kolkata')
current_hour = datetime.now(ist).hour

# ==============================
# SHOW CHART
# ==============================

if 17 <= current_hour < 19:

    color_map = {
        'GAME': 'pink'
    }

    fig = px.scatter(
        df_filtered,
        x='Size',
        y='Rating',
        size='Installs',
        color='Category_Translated',
        hover_name='App',
        title='App Size vs Rating (Bubble Chart)',
        size_max=60,
        color_discrete_map=color_map
    )

    fig.update_layout(
        xaxis_title="App Size (MB)",
        yaxis_title="Average Rating",
        template="plotly_white"
    )

    # ==============================
    # SHOW CHART
    # ==============================

    fig.show()

    # ==============================
    # SAVE HTML
    # ==============================

    task5_file = "Task5_BubbleChart.html"

    fig.write_html(
        os.path.join(html_files_path, task5_file),
        full_html=False,
        include_plotlyjs='cdn'
    )

    # ==============================
    # ADD TO DASHBOARD
    # ==============================

    plot_containers += f"""
    <div class="plot-container" onclick="openPlot('{task5_file}')">

        <iframe
            src="{task5_file}"
            width="100%"
            height="100%"
            frameborder="0">
        </iframe>

        <div class="insights">
            <strong>Task 5 Insights:</strong><br>
            Larger apps with higher installs generally maintain strong ratings.<br>
            Bubble size represents installs while colors indicate categories.
        </div>

    </div>
    """

else:
    print("Chart visible only between 5 PM and 7 PM IST")

Chart visible only between 5 PM and 7 PM IST


In [80]:
# ==============================
# TASK 6 WITH HTML DASHBOARD
# ==============================

import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime
import pytz
import os

df = pd.read_csv('googleplaystore.csv')

# ==============================
# CLEAN INSTALLS
# ==============================

df['Installs'] = df['Installs'].str.replace('[+,]', '', regex=True)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

# ==============================
# CLEAN SIZE
# ==============================

df['Size'] = df['Size'].str.replace('M', '', regex=False)
df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

# ==============================
# CLEAN REVIEWS
# ==============================

df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

# ==============================
# CONVERT DATE
# ==============================

df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

# ==============================
# APPLY FILTERS
# ==============================

df_filtered = df[
    (df['Rating'] >= 4.2) &
    (~df['App'].str.contains(r'\d', na=False)) &
    (df['Category'].str.startswith(('T', 'P'), na=False)) &
    (df['Reviews'] > 1000) &
    (df['Size'].between(20, 80))
].copy()

# ==============================
# CREATE MONTHLY TIME SERIES
# ==============================

df_filtered['Month'] = (
    df_filtered['Last Updated']
    .dt.to_period('M')
    .dt.to_timestamp()
)

# ==============================
# AGGREGATE INSTALLS
# ==============================

monthly_data = (
    df_filtered
    .groupby(['Month', 'Category'])['Installs']
    .sum()
    .reset_index()
)

# ==============================
# TRANSLATE CATEGORIES
# ==============================

translation_map = {
    'TRAVEL_AND_LOCAL': 'Voyage et Local',
    'PRODUCTIVITY': 'Productividad',
    'PHOTOGRAPHY': '写真'
}

monthly_data['Category_Translated'] = (
    monthly_data['Category']
    .replace(translation_map)
)

# ==============================
# CALCULATE GROWTH
# ==============================

monthly_data['Growth'] = (
    monthly_data
    .groupby('Category')['Installs']
    .pct_change()
)

monthly_data['Highlight'] = np.where(
    monthly_data['Growth'] > 0.25,
    'High Growth',
    'Normal'
)

# ==============================
# TIME RESTRICTION
# ==============================

ist = pytz.timezone('Asia/Kolkata')
current_hour = datetime.now(ist).hour

# ==============================
# CREATE CHART
# ==============================

if 16 <= current_hour < 18:

    fig = px.area(
        monthly_data,
        x='Month',
        y='Installs',
        color='Category_Translated',
        title='Cumulative Installs Over Time by Category'
    )

    # Highlight high growth months
    for trace in fig.data:

        cat = trace.name

        highlight_points = monthly_data[
            (monthly_data['Category_Translated'] == cat) &
            (monthly_data['Highlight'] == 'High Growth')
        ]

        if not highlight_points.empty:
            trace.line.width = 4

    fig.update_layout(
        template='plotly_white',
        xaxis_title='Month',
        yaxis_title='Total Installs'
    )

    # ==============================
    # SHOW CHART
    # ==============================

    fig.show()

    # ==============================
    # SAVE HTML
    # ==============================

    task6_file = "Task6_AreaChart.html"

    fig.write_html(
        os.path.join(html_files_path, task6_file),
        full_html=False,
        include_plotlyjs='cdn'
    )

    # ==============================
    # ADD TO DASHBOARD
    # ==============================

    plot_containers += f"""
    <div class="plot-container" onclick="openPlot('{task6_file}')">

        <iframe
            src="{task6_file}"
            width="100%"
            height="100%"
            frameborder="0">
        </iframe>

        <div class="insights">
            <strong>Task 6 Insights:</strong><br>
            Area chart displays cumulative installs over time.<br>
            Thicker lines indicate categories with more than 25% growth.
        </div>

    </div>
    """

else:
    print("Chart visible only between 4 PM and 6 PM IST")

Chart visible only between 4 PM and 6 PM IST


In [81]:
# ==============================
# FINAL DASHBOARD HTML
# ==============================

import os
import webbrowser

# ==========================================
# CHECK IF CHARTS EXIST
# ==========================================

if not plot_containers.strip():

    plot_containers = """
    <div class="no-chart-message">
        <h2>No Charts Available Right Now</h2>
        <p>Please check again after some time.</p>
    </div>
    """

# ==========================================
# DASHBOARD HTML
# ==========================================

dashboard_html = f"""
<!DOCTYPE html>
<html lang="en">

<head>

    <meta charset="UTF-8">

    <meta name="viewport"
          content="width=device-width, initial-scale=1.0">

    <title>Internship Dashboard</title>

    <style>

        body {{
            font-family: Arial, sans-serif;
            background-color: #1e1e1e;
            color: white;
            margin: 0;
            padding: 0;
        }}

        /* ==========================================
           HEADER
        ========================================== */

        .header {{
            background-color: #111;
            padding: 20px;
            text-align: center;
            font-size: 32px;
            font-weight: bold;
            letter-spacing: 1px;
        }}

        .sub-header {{
            text-align: center;
            padding-bottom: 20px;
            color: #bbb;
            font-size: 18px;
        }}

        /* ==========================================
           MAIN CONTAINER
        ========================================== */

        .container {{

            display: grid;

            grid-template-columns: 1fr 1fr;

            gap: 20px;

            padding: 20px;
        }}

        /* ==========================================
           PLOT CONTAINER
        ========================================== */

        .plot-container {{

            background: #2b2b2b;

            border-radius: 12px;

            overflow: hidden;

            box-shadow: 0 0 10px rgba(255,255,255,0.1);

            transition: transform 0.3s ease;

            padding: 10px;
        }}

        .plot-container:hover {{
            transform: scale(1.01);
        }}

        /* ==========================================
           IFRAME
        ========================================== */

        iframe {{

            width: 100%;

            height: 500px;

            border: none;

            background: white;

            border-radius: 10px;
        }}

        /* ==========================================
           INSIGHTS
        ========================================== */

        .insights {{

            margin-top: 10px;

            background: rgba(0,0,0,0.8);

            color: white;

            padding: 12px;

            font-size: 14px;

            border-radius: 8px;

            line-height: 1.5;
        }}

        /* ==========================================
           NO CHART MESSAGE
        ========================================== */

        .no-chart-message {{

            grid-column: span 2;

            text-align: center;

            background: #2b2b2b;

            padding: 50px;

            border-radius: 15px;

            box-shadow: 0 0 10px rgba(255,255,255,0.1);
        }}

        .no-chart-message h2 {{

            color: #ffcc00;

            font-size: 32px;

            margin-bottom: 15px;
        }}

        .no-chart-message p {{

            color: #ccc;

            font-size: 18px;
        }}

        /* ==========================================
           RESPONSIVE
        ========================================== */

        @media screen and (max-width: 1000px) {{

            .container {{
                grid-template-columns: 1fr;
            }}

            .no-chart-message {{
                grid-column: span 1;
            }}
        }}

    </style>

</head>

<body>

    <div class="header">
        Internship Analytics Dashboard
    </div>

    <div class="sub-header">
        Interactive Visualization of All Internship Tasks
    </div>

    <div class="container">

        {plot_containers}

    </div>

</body>

</html>
"""

# ==========================================
# SAVE DASHBOARD
# ==========================================

dashboard_path = os.path.join(
    html_files_path,
    "Internship_Dashboard.html"
)

with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(dashboard_html)

# ==========================================
# OPEN DASHBOARD
# ==========================================

webbrowser.open(
    'file:///' + os.path.realpath(dashboard_path)
)

print("Dashboard Created Successfully!")

Dashboard Created Successfully!
